# Chapter 7 — The Chain Rule and Automatic Differentiation

Companion notebook to Chapter 7 of *Advanced Calculus for Machine Learning*.
Every numerical claim of the chapter is recomputed here with PyTorch and
checked with an assertion.

The two operations of the chapter are

* `jvp(f, (a,), (v,))`: the push-forward `Df(a) v` of a velocity (forward mode);
* `vjp(f, a)`: returns `f(a)` and a function `psi -> Df(a)^T psi`, the
  pull-back of a covector (reverse mode).

In [1]:
import math
import time
import warnings
import torch
from torch.func import jacrev, jacfwd, jvp, vjp

warnings.filterwarnings("ignore", category=FutureWarning)
torch.set_default_dtype(torch.float64)
torch.manual_seed(0)
print(torch.__version__)

2.15.0+cu132


## 1. Jacobian matrices multiply

Theorem 7.1 and Corollary 7.2, on Exercise 7.1: `f(x, y) = (x + y^2, x y)`
and `g(u, v) = (u v, u - v, v^2)` at `(1, 2)`.

In [2]:
f = lambda p: torch.stack([p[0] + p[1] ** 2, p[0] * p[1]])
g = lambda q: torch.stack([q[0] * q[1], q[0] - q[1], q[1] ** 2])
a = torch.tensor([1.0, 2.0])
b = f(a)
Jf, Jg = jacrev(f)(a), jacrev(g)(b)
Jgf = jacrev(lambda p: g(f(p)))(a)
print("f(a) =", b.tolist())
print("J_f =\n", Jf, "\nJ_g =\n", Jg, "\nJ_(g o f) =\n", Jgf)
assert torch.allclose(Jf, torch.tensor([[1.0, 4.0], [2.0, 1.0]]))
assert torch.allclose(Jg, torch.tensor([[2.0, 5.0], [1.0, -1.0], [0.0, 4.0]]))
assert torch.allclose(Jgf, Jg @ Jf)
assert torch.allclose(Jgf, torch.tensor([[12.0, 13.0], [-1.0, 3.0], [8.0, 4.0]]))

# first column by forward mode, third row by reverse mode
col = jvp(lambda p: g(f(p)), (a,), (torch.tensor([1.0, 0.0]),))[1]
_, pull = vjp(lambda p: g(f(p)), a)
(row,) = pull(torch.tensor([0.0, 0.0, 1.0]))
print("first column (forward):", col.tolist(), "  third row (reverse):", row.tolist())
assert torch.allclose(col, Jgf[:, 0]) and torch.allclose(row, Jgf[2])

f(a) = [5.0, 2.0]
J_f =
 tensor([[1., 4.],
        [2., 1.]]) 
J_g =
 tensor([[ 2.,  5.],
        [ 1., -1.],
        [ 0.,  4.]]) 
J_(g o f) =
 tensor([[12., 13.],
        [-1.,  3.],
        [ 8.,  4.]])
first column (forward): [12.0, -1.0, 8.0]   third row (reverse): [8.0, 4.0]


## 2. Push-forward, pull-back and the pairing

Equation (7.1): `(Df(a)^T psi)(v) = psi(Df(a) v)`. The two modes compute the
same number in two different ways; this identity is the standard test of an
automatic differentiation library.

In [3]:
torch.manual_seed(1)
A1, A2 = torch.randn(7, 5), torch.randn(4, 7)
net = lambda x: torch.tanh(A2 @ torch.sin(A1 @ x))      # R^5 -> R^4
x = torch.randn(5); v = torch.randn(5); psi = torch.randn(4)

_, push = jvp(net, (x,), (v,))
_, pull = vjp(net, x)
(pulled,) = pull(psi)
lhs, rhs = pulled @ v, psi @ push
print("(Df^T psi)(v) =", lhs.item(), "   psi(Df v) =", rhs.item())
assert torch.allclose(lhs, rhs)

J = jacrev(net)(x)
assert torch.allclose(push, J @ v)          # column J [v]
assert torch.allclose(pulled, psi @ J)      # row [psi] J
assert torch.allclose(pulled, J.T @ psi)    # the same numbers, stored as a column

# Proposition 7.6(3): d(h o f) = Df^T dh, on rows [dh] J_f
h = lambda y: (y ** 2).sum()
row_h = jacrev(h)(net(x))
assert torch.allclose(jacrev(lambda x: h(net(x)))(x), row_h @ J)
print("d(h o f)(a) equals the pull-back of dh(f(a))")

(Df^T psi)(v) = 0.10942043238748983    psi(Df v) = 0.10942043238748984
d(h o f)(a) equals the pull-back of dh(f(a))


## 3. The examples of Section 7.1

The norm of an inner product space, Jacobi's formula (Exercise 7.4), least
squares, and the normalization map of Exercise 7.3.

In [4]:
M = torch.tensor([[2.0, 0.5, 0.0], [0.5, 1.0, 0.2], [0.0, 0.2, 3.0]])   # Gram matrix
ip = lambda u, w: u @ M @ w
norm = lambda u: torch.sqrt(ip(u, u))
x = torch.tensor([1.0, -2.0, 0.5]); hh = torch.tensor([0.3, 0.1, -0.4])

assert torch.allclose(jvp(norm, (x,), (hh,))[1], ip(x, hh) / norm(x))

# normalization n(x) = x / |x|: Dn(x)h = (h - <u,h>u)/|x|
n = lambda u: u / norm(u)
u = n(x)
d = jvp(n, (x,), (hh,))[1]
assert torch.allclose(d, (hh - ip(u, hh) * u) / norm(x))
assert torch.allclose(jvp(n, (x,), (x,))[1], torch.zeros(3), atol=1e-14)   # kernel
assert abs(ip(u, d).item()) < 1e-14                                         # image
print("Dn(x) kills x, and its values are orthogonal to x")

# Jacobi's formula on A(t) = A0 + t I
A0 = torch.tensor([[2.0, 1.0], [1.0, 1.0]])
d = jvp(lambda t: torch.linalg.det(A0 + t * torch.eye(2)),
        (torch.tensor(0.0),), (torch.tensor(1.0),))[1]
print("d/dt det(A0 + tI) at 0:", d.item())
assert abs(d.item() - 3.0) < 1e-12
assert abs((torch.linalg.det(A0) * torch.trace(torch.linalg.inv(A0))).item() - 3.0) < 1e-12

# least squares with Gram matrices M (parameters) and N (data)
torch.manual_seed(2)
A = torch.randn(4, 3); bvec = torch.randn(4)
N = torch.diag(torch.tensor([1.0, 2.0, 0.5, 3.0]))
loss = lambda z: 0.5 * (A @ z - bvec) @ N @ (A @ z - bvec)
row = jacrev(loss)(x)
Astar = torch.linalg.solve(M, A.T @ N)                  # adjoint of A
assert torch.allclose(row, (A @ x - bvec) @ N @ A)      # pull-back of dq by A
assert torch.allclose(row, M @ (Astar @ (A @ x - bvec)))  # = R_V(A*(Ax - b))
print("d f(x) is the Riesz image of A*(Ax - b)")

Dn(x) kills x, and its values are orthogonal to x
d/dt det(A0 + tI) at 0: 3.0
d f(x) is the Riesz image of A*(Ax - b)


## 4. Both modes by hand

Example 7.8 and Figure 7.3: `f1(x) = (x1 x2, x1 - x2)`, `f2(u) = u1^2 + 3 u2`
at `(2, 1)`.

In [5]:
f1 = lambda p: torch.stack([p[0] * p[1], p[0] - p[1]])
f2 = lambda q: q[0] ** 2 + 3 * q[1]
a0 = torch.tensor([2.0, 1.0])
a1 = f1(a0); a2 = f2(a1)
J1, J2 = jacrev(f1)(a0), jacrev(f2)(a1)
print("a1 =", a1.tolist(), " a2 =", a2.item())
print("J1 =", J1.tolist(), " J2 =", J2.tolist())
assert torch.allclose(J1, torch.tensor([[1.0, 2.0], [1.0, -1.0]]))
assert torch.allclose(J2, torch.tensor([4.0, 3.0])) and a2.item() == 7.0

# forward mode: two passes
for v0, expected in ((torch.tensor([1.0, 0.0]), 7.0), (torch.tensor([0.0, 1.0]), 5.0)):
    v1 = J1 @ v0; v2 = J2 @ v1
    print("forward:  v0 =", v0.tolist(), " v1 =", v1.tolist(), " v2 =", v2.item())
    assert v2.item() == expected

# reverse mode: one pass
psi2 = torch.tensor(1.0); psi1 = psi2 * J2; psi0 = psi1 @ J1
print("reverse:  psi1 =", psi1.tolist(), " psi0 =", psi0.tolist())
assert torch.allclose(psi0, torch.tensor([7.0, 5.0]))

# the same on the graph of Figure 7.3, node by node
x1, x2 = 2.0, 1.0
p = x1 * x2; d = x1 - x2; s = p ** 2; t = 3 * d; y = s + t
ybar = 1.0; sbar = ybar; tbar = ybar            # sum: the covector is copied
pbar = 2 * p * sbar; dbar = 3 * tbar
x1bar = x2 * pbar + dbar                        # x1 is used twice: add
x2bar = x1 * pbar - dbar
print("graph:    x1bar =", x1bar, " x2bar =", x2bar)
assert (y, pbar, dbar, x1bar, x2bar) == (7.0, 4.0, 3.0, 7.0, 5.0)

xx = a0.clone().requires_grad_(True)
(auto,) = torch.autograd.grad(f2(f1(xx)), xx)
assert torch.allclose(auto, torch.tensor([7.0, 5.0]))

a1 = [2.0, 1.0]  a2 = 7.0
J1 = [[1.0, 2.0], [1.0, -1.0]]  J2 = [4.0, 3.0]
forward:  v0 = [1.0, 0.0]  v1 = [1.0, 1.0]  v2 = 7.0
forward:  v0 = [0.0, 1.0]  v1 = [2.0, -1.0]  v2 = 5.0
reverse:  psi1 = [4.0, 3.0]  psi0 = [7.0, 5.0]
graph:    x1bar = 7.0  x2bar = 5.0


## 5. One pass per column, one pass per row

Proposition 7.9. For a function with many inputs and one output, reverse
mode needs one pass and forward mode one pass per input. The counts of
Exercise 7.6 are reproduced, and the two ways of building a Jacobian matrix
(`jacfwd`, column by column, and `jacrev`, row by row) are timed.

In [6]:
# Exercise 7.6: multiplications for J3 (J2 J1) and (J3 J2) J1
dims = [1000, 100, 100, 1]                      # d0, d1, d2, d3
cost = lambda p, q, r: p * q * r
forward_order = cost(100, 100, 1000) + cost(1, 100, 1000)
reverse_order = cost(1, 100, 100) + cost(1, 100, 1000)
print("J3 (J2 J1):", forward_order, "   (J3 J2) J1:", reverse_order,
      "   ratio:", round(forward_order / reverse_order, 1))
assert forward_order == 10_100_000 and reverse_order == 110_000

torch.manual_seed(3)
W1, W2, W3 = torch.randn(100, 1000) / 30, torch.randn(100, 100) / 10, torch.randn(1, 100) / 10
scalar_fn = lambda x: (W3 @ torch.tanh(W2 @ torch.tanh(W1 @ x))).squeeze()
x = torch.randn(1000)

t0 = time.perf_counter(); row_rev = jacrev(scalar_fn)(x); t_rev = time.perf_counter() - t0
t0 = time.perf_counter(); row_fwd = jacfwd(scalar_fn)(x); t_fwd = time.perf_counter() - t0
assert torch.allclose(row_rev, row_fwd)
print(f"1000 inputs, 1 output:  jacrev {t_rev*1e3:.1f} ms   jacfwd {t_fwd*1e3:.1f} ms")

# the opposite shape: 1 input, 1000 outputs
curve = lambda t: torch.tanh(W1.T @ torch.tanh(W2 @ (W3.T.squeeze() * t)))
t = torch.tensor(0.3)
t0 = time.perf_counter(); col_fwd = jacfwd(curve)(t); t_fwd2 = time.perf_counter() - t0
t0 = time.perf_counter(); col_rev = jacrev(curve)(t); t_rev2 = time.perf_counter() - t0
assert torch.allclose(col_fwd, col_rev)
print(f"1 input, 1000 outputs:  jacfwd {t_fwd2*1e3:.1f} ms   jacrev {t_rev2*1e3:.1f} ms")
print("(timings depend on the machine and are not asserted)")

J3 (J2 J1): 10100000    (J3 J2) J1: 110000    ratio: 91.8
1000 inputs, 1 output:  jacrev 0.8 ms   jacfwd 12.4 ms
1 input, 1000 outputs:  jacfwd 1.5 ms   jacrev 7.7 ms
(timings depend on the machine and are not asserted)


## 6. Backpropagation

Theorem 7.10, implemented with the recursion
`delta_{k-1} = delta_k W_k diag(s'(z_{k-1}))` and compared with
`autograd`. The array for `W_k` is the outer product `delta_k^T a_{k-1}^T`.

In [7]:
torch.manual_seed(4)
dims = [6, 5, 4, 3]                              # d0, ..., d3
Ws = [torch.randn(dims[k + 1], dims[k]) for k in range(3)]
bs = [torch.randn(dims[k + 1]) for k in range(3)]
x = torch.randn(dims[0]); target = torch.randn(dims[-1])
s, ds = torch.tanh, lambda z: 1 - torch.tanh(z) ** 2
ell = lambda out: 0.5 * ((out - target) ** 2).sum()

def forward(Ws, bs):
    a, zs, acts = x, [], [x]
    for k, (W, b) in enumerate(zip(Ws, bs)):
        z = W @ a + b
        a = s(z) if k < len(Ws) - 1 else z       # last layer: identity
        zs.append(z); acts.append(a)
    return zs, acts

# forward pass, then the backward recursion on rows
zs, acts = forward(Ws, bs)
delta = acts[-1] - target                        # [d ell(a_N)], last sigma = identity
dW, db = [None] * 3, [None] * 3
for k in reversed(range(3)):
    db[k] = delta
    dW[k] = torch.outer(delta, acts[k])          # delta_k^T a_{k-1}^T
    if k > 0:
        delta = (delta @ Ws[k]) * ds(zs[k - 1])  # delta_k W_k diag(s'(z_{k-1}))

# autograd
Wg = [W.clone().requires_grad_(True) for W in Ws]
bg = [b.clone().requires_grad_(True) for b in bs]
L = ell(forward(Wg, bg)[1][-1])
L.backward()
for k in range(3):
    assert torch.allclose(dW[k], Wg[k].grad) and torch.allclose(db[k], bg[k].grad)
    assert torch.linalg.matrix_rank(dW[k]).item() == 1
print("manual backpropagation agrees with autograd on all 6 parameter tensors")
print("each dL/dW_k has rank", torch.linalg.matrix_rank(dW[0]).item())

# the theorem as a statement about the differential: dL(theta)(H, beta)
Hs = [torch.randn_like(W) for W in Ws]; betas = [torch.randn_like(b) for b in bs]
pairing = sum(db[k] @ (Hs[k] @ acts[k] + betas[k]) for k in range(3))
directional = jvp(lambda *th: ell(forward(th[:3], th[3:])[1][-1]),
                  (*Ws, *bs), (*Hs, *betas))[1]
print("sum_k delta_k(H_k a_{k-1} + beta_k) =", pairing.item(),
      "  forward-mode dL(theta)(H, beta) =", directional.item())
assert torch.allclose(pairing, directional)

manual backpropagation agrees with autograd on all 6 parameter tensors
each dL/dW_k has rank 1
sum_k delta_k(H_k a_{k-1} + beta_k) = -9.154264574765133   forward-mode dL(theta)(H, beta) = -9.15426457476514


In [8]:
# Exercise 7.7 by hand and by autograd
w1 = torch.tensor(1.0, requires_grad=True)
b1 = torch.tensor(1.0, requires_grad=True)
w2 = torch.tensor(0.5, requires_grad=True)
xx, yy = 1.0, 1.0
z1 = w1 * xx + b1; a1 = z1 ** 2; yhat = w2 * a1; L = 0.5 * (yhat - yy) ** 2
L.backward()
print("z1, a1, yhat, L =", z1.item(), a1.item(), yhat.item(), L.item())
print("dL/dw1, dL/db1, dL/dw2 =", w1.grad.item(), b1.grad.item(), w2.grad.item())
assert (z1.item(), a1.item(), yhat.item(), L.item()) == (2.0, 4.0, 2.0, 0.5)
assert (w1.grad.item(), b1.grad.item(), w2.grad.item()) == (2.0, 2.0, 4.0)

z1, a1, yhat, L = 2.0 4.0 2.0 0.5
dL/dw1, dL/db1, dL/dw2 = 2.0 2.0 4.0


### What `backward()` returns is a covector

Store the same model in another basis of parameter space, `theta = P theta'`.
The array returned by `autograd` in the new coordinates is `[dL] P` (the rule
for covectors, Proposition 6.19), not `P^{-1}` times the old array (the rule
for vectors). And the plain optimizer step, which subtracts the array from
the coordinates, does not describe the same displacement in the two bases.

In [9]:
torch.manual_seed(5)
theta = torch.randn(4)
X = torch.randn(10, 4); yv = torch.randn(10)
loss = lambda th: 0.5 * ((torch.tanh(X @ th) - yv) ** 2).mean()
P = torch.randn(4, 4) + 2 * torch.eye(4)
theta_new = torch.linalg.solve(P, theta)

row_old = jacrev(loss)(theta)
row_new = jacrev(lambda t: loss(P @ t))(theta_new)
assert torch.allclose(row_new, row_old @ P)                          # covector rule
assert not torch.allclose(row_new, torch.linalg.solve(P, row_old))   # not the vector rule
print("backward() in the new basis returns [dL] P")

eta = 0.1
step_old = theta - eta * row_old
step_new = P @ (theta_new - eta * row_new)       # the new-basis step, back in old coordinates
print("same point after one optimizer step?", torch.allclose(step_old, step_new))
assert torch.allclose(step_new, theta - eta * P @ P.T @ row_old)     # Proposition 2.24
assert not torch.allclose(step_old, step_new)

backward() in the new basis returns [dL] P
same point after one optimizer step? False


## 7. Vanishing, exploding and residual connections

The covector is multiplied by one matrix per layer. Its norm is bounded by
the product of the largest singular values, and a residual layer passes it
back unchanged plus a correction.

In [10]:
torch.manual_seed(6)
depth, width = 30, 20
x0 = torch.randn(width); psi_out = torch.randn(width)

def backward_norms(scale, residual=False):
    Wl = [scale * torch.randn(width, width) / math.sqrt(width) for _ in range(depth)]
    a, zs = x0, []
    for W in Wl:                                  # forward pass
        z = W @ a; zs.append(z)
        a = a + torch.tanh(z) if residual else torch.tanh(z)
    psi, norms, bound = psi_out, [psi_out.norm().item()], psi_out.norm().item()
    for W, z in zip(reversed(Wl), reversed(zs)):  # pull-back, layer by layer
        Jt = (psi * (1 - torch.tanh(z) ** 2)) @ W
        psi = psi + Jt if residual else Jt
        bound *= torch.linalg.matrix_norm(W, ord=2).item() + (1.0 if residual else 0.0)
        norms.append(psi.norm().item())
        assert norms[-1] <= bound * (1 + 1e-12)
    return norms

for scale in (0.5, 1.0, 3.0):
    nm = backward_norms(scale)
    print(f"plain, weight scale {scale}: |psi| from {nm[0]:.2e} to {nm[-1]:.2e}")
nm_small = backward_norms(0.5)
nm_res = backward_norms(0.5, residual=True)
print(f"residual, weight scale 0.5: |psi| from {nm_res[0]:.2e} to {nm_res[-1]:.2e}")
assert nm_small[-1] < 1e-6 * nm_small[0] and nm_res[-1] > 1e-2 * nm_res[0]

plain, weight scale 0.5: |psi| from 5.57e+00 to 1.52e-09
plain, weight scale 1.0: |psi| from 5.57e+00 to 1.58e+00
plain, weight scale 3.0: |psi| from 5.57e+00 to 1.67e+03
residual, weight scale 0.5: |psi| from 5.57e+00 to 3.86e+01


## 8. Graphs: copying and adding are dual

Lemma 7.11 and Proposition 7.12 (a value used twice), Exercises 7.8 and 7.9,
and the three faces of the rule in PyTorch: weight sharing, accumulation in
`.grad`, and `detach()`.

In [11]:
# a value used twice: g(x) = G(x, x), dg = d1 G + d2 G
G = lambda w1, w2, x: w2 * torch.tanh(w1 * x)
w, x = torch.tensor(0.8), torch.tensor(1.3)
d1 = jacrev(G, argnums=0)(w, w, x); d2 = jacrev(G, argnums=1)(w, w, x)
dg = jacrev(lambda w: G(w, w, x))(w)
print("d1 G + d2 G =", (d1 + d2).item(), "   dg =", dg.item())
assert torch.allclose(dg, d1 + d2)
assert torch.allclose(dg, w * x * (1 - torch.tanh(w * x) ** 2) + torch.tanh(w * x))

# Exercise 7.8: y = (x1 + x2) x1 at (3, 2)
xs = torch.tensor([3.0, 2.0], requires_grad=True)
y = (xs[0] + xs[1]) * xs[0]
y.backward()
print("Exercise 7.8: y =", y.item(), " [xbar1, xbar2] =", xs.grad.tolist())
assert xs.grad.tolist() == [8.0, 3.0]

# the dual of the sum is the copy
u = torch.tensor([1.0, 2.0], requires_grad=True); v = torch.tensor([3.0, -1.0], requires_grad=True)
psi = torch.tensor([0.7, -0.2])
(psi @ (u + v)).backward()
assert torch.allclose(u.grad, psi) and torch.allclose(v.grad, psi)

# weight sharing in a recurrent map: contributions of the time steps add
torch.manual_seed(7)
Wr = torch.randn(3, 3) / 2; h0 = torch.randn(3)
def unrolled(W_list):
    h = h0
    for W in W_list:
        h = torch.tanh(W @ h)
    return h.sum()
copies = [Wr.clone().requires_grad_(True) for _ in range(4)]
unrolled(copies).backward()
shared = Wr.clone().requires_grad_(True)
unrolled([shared] * 4).backward()
assert torch.allclose(shared.grad, sum(c.grad for c in copies))
print("shared weight: its array is the sum of the arrays of the 4 time steps")

d1 G + d2 G = 1.1885738286890666    dg = 1.1885738286890666
Exercise 7.8: y = 15.0  [xbar1, xbar2] = [8.0, 3.0]
shared weight: its array is the sum of the arrays of the 4 time steps


In [12]:
# accumulation: backward() adds to .grad
w = torch.tensor([1.0, 2.0], requires_grad=True)
L1 = lambda w: (w ** 2).sum(); L2 = lambda w: (3 * w).sum()
L1(w).backward(); first = w.grad.clone()
L2(w).backward(); both = w.grad.clone()
print("after L1:", first.tolist(), "  after L1 then L2:", both.tolist())
assert torch.allclose(both, jacrev(lambda w: L1(w) + L2(w))(w.detach()))
w.grad.zero_()
L2(w).backward()
assert torch.allclose(w.grad, torch.tensor([3.0, 3.0]))

# detach: a different function is differentiated
w = torch.tensor(0.8, requires_grad=True)
(w.detach() * torch.tanh(w * 1.3)).backward()
print("with the outer factor detached:", w.grad.item(), "  (= d1 G only:", d1.item(), ")")
assert torch.allclose(w.grad, d1)

# residual layer: pull-back is psi + DF^T psi
F = lambda x: 0.1 * torch.sin(x[[1, 2, 0]])
x = torch.randn(3); psi = torch.randn(3)
(pulled,) = vjp(lambda x: x + F(x), x)[1](psi)
assert torch.allclose(pulled, psi + vjp(F, x)[1](psi)[0])

after L1: [2.0, 4.0]   after L1 then L2: [5.0, 7.0]
with the outer factor detached: 0.41068576211188157   (= d1 G only: 0.41068576211188157 )


## Summary

* Jacobian matrices of a composition multiply; forward mode multiplies from
  the right with a column, reverse mode from the left with a row.
* `jvp` and `vjp` are linked by `(Df^T psi)(v) = psi(Df v)`.
* Backpropagation is the reverse recursion on rows; the array for a weight
  matrix is an outer product; the result transforms as a covector.
* The plain optimizer step is not the same displacement in two bases.
* On a graph, contributions add wherever a value is used several times.